# Module 05 — Lesson 07
# GroupBy

---

> Every lesson so far has looked at rows one at a time -- filter these, sort those. But the question "what's the average tip **by day**?" isn't about any single row; it's about whole categories at once. That's exactly what `.groupby()` is for: split the data into buckets by category, summarize each bucket, then line the summaries back up into one table. This pattern is called **split-apply-combine**, and it's arguably the single most useful tool in all of pandas.

We're back to `tips.csv` for this lesson -- grouping by `day`, `time`, and `smoker` gives real, meaningful categories to summarize.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Explain the split-apply-combine pattern that `.groupby()` implements
- Group by one column and summarize with a single aggregation (`.mean()`, `.sum()`, `.count()`, `.size()`)
- Group by multiple columns at once
- Apply several aggregations in one call with `.agg()`, including different functions per column
- Turn a grouped result back into a normal flat DataFrame with `.reset_index()`

In [1]:
import pandas as pd

tips = pd.read_csv("data/tips.csv")
print(f"tips.shape: {tips.shape}")
print(tips.head())

tips.shape: (244, 7)
   total_bill   tip     sex smoker  day    time  size
0       16.99  1.01  Female     No  Sun  Dinner     2
1       10.34  1.66    Male     No  Sun  Dinner     3
2       21.01  3.50    Male     No  Sun  Dinner     3
3       23.68  3.31    Male     No  Sun  Dinner     2
4       24.59  3.61  Female     No  Sun  Dinner     4


---
## 1. Split-Apply-Combine, in One Line

`tips.groupby('day')` alone doesn't compute anything yet -- it just describes how to **split** the rows into groups (one per unique value of `day`). You need to chain an aggregation to **apply** a calculation to each group and **combine** the results into a summary table.

In [2]:
grouped = tips.groupby("day")
print(f"type: {type(grouped)}")   # not a DataFrame yet -- just a grouping plan
print()

avg_bill_by_day = grouped["total_bill"].mean()
print("Average total_bill per day:")
print(avg_bill_by_day)
print(f"\ntype of result: {type(avg_bill_by_day)}")

type: <class 'pandas.api.typing.DataFrameGroupBy'>

Average total_bill per day:
day
Fri     17.151579
Sat     20.441379
Sun     21.410000
Thur    17.682742
Name: total_bill, dtype: float64

type of result: <class 'pandas.Series'>


---
## 2. Common Single-Column Aggregations

Once grouped, almost any Series method works per-group: `.mean()`, `.sum()`, `.count()`, `.min()`, `.max()`, `.std()`.

In [3]:
print("Total tip $ collected per day:")
print(tips.groupby("day")["tip"].sum())
print()

print("Largest single bill per day:")
print(tips.groupby("day")["total_bill"].max())

Total tip $ collected per day:
day
Fri      51.96
Sat     260.40
Sun     247.39
Thur    171.83
Name: tip, dtype: float64

Largest single bill per day:
day
Fri     40.17
Sat     50.81
Sun     48.17
Thur    43.11
Name: total_bill, dtype: float64


---
## 3. `.size()` vs `.count()` — Not the Same Thing

`.size()` counts **rows** per group, full stop. `.count()` counts **non-null values**, column by column -- on a dataset with no missing values (like `tips`) they agree, but they diverge the moment a column has gaps.

In [4]:
print("size() -- rows per day (one number per group):")
print(tips.groupby("day").size())
print()

print("count() -- non-null values per COLUMN, per day:")
print(tips.groupby("day").count())

size() -- rows per day (one number per group):
day
Fri     19
Sat     87
Sun     76
Thur    62
dtype: int64

count() -- non-null values per COLUMN, per day:
      total_bill  tip  sex  smoker  time  size
day                                           
Fri           19   19   19      19    19    19
Sat           87   87   87      87    87    87
Sun           76   76   76      76    76    76
Thur          62   62   62      62    62    62


In [6]:
# To see them actually DIFFER, bring back planets.csv from Lesson 06,
# which has real missing values in 'mass'
planets = pd.read_csv("data/planets.csv")
print(planets.head())
size_per_method = planets.groupby("method").size()
mass_count_per_method = planets.groupby("method")["mass"].count()
comparison = pd.DataFrame({
    "size (all rows)": size_per_method,
    "count (non-null mass)": mass_count_per_method,
})
print(comparison)

            method  number  orbital_period   mass  distance  year
0  Radial Velocity       1         269.300   7.10     77.40  2006
1  Radial Velocity       1         874.774   2.21     56.95  2008
2  Radial Velocity       1         763.000   2.60     19.84  2011
3  Radial Velocity       1         326.030  19.40    110.62  2007
4  Radial Velocity       1         516.220  10.50    119.47  2009
                               size (all rows)  count (non-null mass)
method                                                               
Astrometry                                   2                      0
Eclipse Timing Variations                    9                      2
Imaging                                     38                      0
Microlensing                                23                      0
Orbital Brightness Modulation                3                      0
Pulsar Timing                                5                      0
Pulsation Timing Variations                 

---
## 4. Grouping by Multiple Columns

Pass a **list** of column names to group by more than one category at once -- pandas creates one group per unique *combination*.

In [ ]:
avg_tip_by_day_time = tips.groupby(["day", "time"])["tip"].mean()
print("Average tip by day AND time:")
print(avg_tip_by_day_time)

Notice the result has a **MultiIndex** -- `day` and `time` are both baked into the index now, not regular columns. `.reset_index()` turns them back into normal columns, which you'll usually want before filtering or sorting the result further.

In [ ]:
avg_tip_flat = avg_tip_by_day_time.reset_index()
print("After reset_index() -- a normal flat DataFrame:")
print(avg_tip_flat)
print(f"\ntype: {type(avg_tip_flat)}")

---
## 5. Multiple Aggregations at Once: `.agg()`

`.agg()` applies more than one function in a single call -- either the same functions across one column, or a different set of functions per column via a dict.

In [ ]:
# Multiple functions on ONE column
bill_stats_by_day = tips.groupby("day")["total_bill"].agg(["mean", "max", "count"])
print("Mean, max, and count of total_bill, per day:")
print(bill_stats_by_day)

In [ ]:
# Different functions for DIFFERENT columns, via a dict
summary_by_day = tips.groupby("day").agg({
    "total_bill": "mean",
    "tip": "max",
    "size": "sum",
})
print("Custom per-column aggregation:")
print(summary_by_day)

---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Expecting groupby() alone to return a usable table --------------

print("Mistake 1 — a groupby object isn't a DataFrame until you aggregate it:")
grouped = tips.groupby("day")
print(f"  print(grouped) just shows: {grouped}")
print()
print("  Correct: chain an aggregation before you try to read or print it:")
print(grouped["tip"].mean())

In [ ]:
# -- Mistake 2: Assuming .size() and .count() always agree --------------------

print("Mistake 2 — .size() and .count() can disagree when a column has NaNs:")
planets = pd.read_csv("data/planets.csv")
size_per_method = planets.groupby("method").size()
mass_count_per_method = planets.groupby("method")["mass"].count()

comparison = pd.DataFrame({"size (all rows)": size_per_method, "count (non-null mass)": mass_count_per_method})
print(comparison)
print()
print("  'Transit' has 397 rows but only 1 non-null 'mass' value --")
print("  using count() when you meant size() would make that method look almost")
print("  nonexistent instead of the second-most-common one in the dataset.")

In [ ]:
# -- Mistake 3: Forgetting a grouped result has a (Multi)Index, not columns -----

print("Mistake 3 — trying to filter a grouped result like a normal DataFrame fails:")
avg_by_day = tips.groupby("day")["total_bill"].mean()
try:
    avg_by_day[avg_by_day["total_bill"] > 18]   # 'day' is the INDEX, not a column
except Exception as e:
    print(f"  Caught error: {type(e).__name__}: {e}")
print()
print("  Correct: reset_index() first, so 'day' becomes a normal column again:")
avg_by_day_flat = avg_by_day.reset_index()
print(avg_by_day_flat[avg_by_day_flat["total_bill"] > 18])

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Single Column, Single Aggregation

Group `tips` by `"time"` and compute the mean `"tip"` for each group.

In [ ]:
# Your code here

### Exercise 2 — `.size()`

Group `tips` by `"time"` and use `.size()` to count how many rows fall into each group.

In [ ]:
# Your code here

### Exercise 3 — Multiple Columns

Group `tips` by `"day"` and `"smoker"` together, and compute the mean `"total_bill"` for each combination. Print the result.

In [ ]:
# Your code here

### Exercise 4 — `.agg()`

Group `tips` by `"day"` and use `.agg()` to compute both the mean and the maximum of `"total_bill"` in one call.

In [ ]:
# Your code here

### Exercise 5 — (Challenge) Group, Reset, Sort

Group `tips` by `"day"`, compute the mean `"tip"`, reset the index so `"day"` is a normal column again, then sort the result to find which day has the **highest** average tip.

In [ ]:
# Your code here

---
## 📌 Key Takeaways

- **`.groupby()` implements split-apply-combine** — it splits rows into groups by one or more columns, you apply an aggregation to summarize each group, and the results combine into a new table. The groupby object itself isn't usable until you chain an aggregation.

- **`.size()` counts rows per group; `.count()` counts non-null values per column** — they agree on clean data but diverge the moment a column has missing values, so pick deliberately.

- **`.agg()` applies multiple functions at once** — a list for several stats on one column, or a dict for different stats per column — and a grouped result needs `.reset_index()` before you can filter or sort it like a normal DataFrame again.

---
## What's Next?

**Lesson 08 — Merge and Join** takes two separate tables — like a summary from `groupby()` and the original data — and combines them side by side.